# PhoBERT v5: consistent labels + text fix + active learning

Setup: **GPU T4 x2**, **Internet On**, secret `HF_TOKEN`.
**Input:** the same private datasets as v4: the three `vfyr_*_cleaned.csv` files **and** `active_labeled_v4.csv`.

Changes from v4 (data only, same model and hyper-parameters):
1. `relabels/guidelines_v31.csv`: 790 comments with body shaming, insults with intensifiers and "cút"
   re-labeled under guideline v3.1 (163 labels changed, train/val/test).
2. `scripts/fix_bad_token.py`: "bad" -> "đéo" (a thesis-era text artifact in ~1,700 comments).
3. Active-learning batch added to train (x3), as in v4.

The model is pushed to the **`v5` branch** (`main` is untouched).


In [ ]:
!git clone -q https://github.com/Harukaru1/vietnamese-toxic-comment-moderation.git /kaggle/working/repo
%cd /kaggle/working/repo
!pip install -q pyvi emoji cleanlab
import glob, os, shutil, json
os.makedirs("data/raw", exist_ok=True)
csvs = [p for p in glob.glob("/kaggle/input/**/*.csv", recursive=True) if "active_labeled" not in os.path.basename(p)]
for split in ["train", "val", "test"]:
    matches = [p for p in csvs if split in os.path.basename(p).lower()]
    assert len(matches) == 1, f"Expected exactly one CSV for '{split}', found: {matches}"
    shutil.copy(matches[0], f"data/raw/{split}.csv")
os.makedirs("reports", exist_ok=True)
!python scripts/find_label_issues.py | tail -2
!python scripts/make_clean_splits.py --prune > /dev/null
!python scripts/add_vihsd.py > /dev/null
!python scripts/apply_relabels.py --src data/clean_vihsd --dst data/clean_vihsd_v3
active = glob.glob("/kaggle/input/**/active_labeled_v4.csv", recursive=True)
assert len(active) == 1, f"Upload active_labeled_v4.csv as a Kaggle dataset, found: {active}"
os.makedirs("data/active", exist_ok=True)
shutil.copy(active[0], "data/active/active_labeled_v4.csv")
!python scripts/apply_relabels.py --src data/clean_vihsd_v3 --dst data/clean_vihsd_v31 --patch relabels/guidelines_v31.csv
!python scripts/fix_bad_token.py --dir data/clean_vihsd_v31
!python scripts/add_active_labels.py --active data/active/active_labeled_v4.csv --src data/clean_vihsd_v31 --dst data/clean_vihsd_v5 --report reports/add_active_v5.json


In [ ]:
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
from huggingface_hub import whoami
print("Logged in as:", whoami()["name"])

In [ ]:
!python scripts/train_phobert.py --data-dir data/clean_vihsd_v5 --no-class-weights --epochs 6 --output-dir models/v5 --tag _v5 --no-push


In [ ]:
from huggingface_hub import HfApi
api = HfApi()
repo_id = f"{whoami()['name']}/phobert-vi-toxic-comment"
api.create_branch(repo_id, branch="v5", exist_ok=True)
api.upload_folder(folder_path="models/v5/best", repo_id=repo_id, revision="v5", commit_message="v5: guideline v3.1 relabels + bad-token fix + active batch")
print(f"https://huggingface.co/{repo_id}/tree/v5")

Copy the output of the next cell and paste it into the chat.

In [ ]:
print(json.dumps(json.load(open("reports/phobert_results_v5.json")), ensure_ascii=False))
